# I22 USAXS processing through the MoDaCor runtime

This working notebook preprocesses each four-scan USAXS acquisition, reuses scans 977724--977727 as the empty-furnace background, and executes the correction graph in `pipelines/I22_USAXS.yaml`. The implementation rationale and open calibration choices are recorded in [`USAXS_PROCESSING_DESIGN.md`](USAXS_PROCESSING_DESIGN.md).


## Imports and example discovery


In [ ]:
from pathlib import Path
import atexit
import sys

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "example_utils.py").is_file():
        EXAMPLES_ROOT = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter from MoDaCor_examples or one of its subdirectories.")

sys.path.insert(0, str(EXAMPLES_ROOT))
from example_utils import locate_example_dir

PROJECT_DIR = locate_example_dir("DLS/I22")
sys.path.insert(0, str(PROJECT_DIR))

import h5py
import hdf5plugin
import matplotlib.pyplot as plt
import modacor
import numpy as np
from IPython.display import JSON, Markdown, display
from modacor.client import LocalRuntimeServer
from modacor.runner.pipeline import Pipeline

from usaxs_preprocess import (
    BACKGROUND_SCAN_SET,
    SAMPLE_SCAN_SETS,
    preprocess_usaxs_acquisition,
)


## Configuration

`MAX_MEASUREMENTS = 1` is useful for a quick smoke run. The high-gain front threshold and fit interval remain provisional, documented pipeline parameters.


In [ ]:
DATA_DIR = PROJECT_DIR / "data" / "usaxs"
PREPROCESSED_DIR = PROJECT_DIR / "work" / "preprocessed" / "usaxs"
OUTPUT_DIR = PROJECT_DIR / "work" / "output" / "usaxs"
PIPELINE_PATH = PROJECT_DIR / "pipelines" / "I22_USAXS.yaml"
MAX_MEASUREMENTS = None
OVERWRITE_PREPROCESSED = False

SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8901
TRACE = {"enabled": True, "watch": {"pooled": ["signal", "Q"]}}
OUTPUT_DATA_PATHS = [
    "/pooled/signal",
    "/pooled/Q",
    "/pooled/bin_id",
    "/pooled/bin_edges",
    "/pooled/bin_count",
    "/pooled/effective_sample_size",
    "/pooled/positive_weight_count",
    "/pooled/sum_weights",
    "/sample_integral/signal",
    "/sample_rear_integral/signal",
    "/SLF/beam_center",
    "/SHF/beam_center",
    "/SLF/signed_Q",
    "/SLF/both_wings_signal",
    "/SLR/scale_factor",
    "/SHF/scale_factor",
    "/SHR/scale_factor",
]


## Lean preprocessing

The preprocessor reduces diode subreads and subtracts the matching exposure-scaled diode dark. I0 is summarized but receives no dark correction. MoDaCor performs count-time normalization, dynamic masking, centering, Q conversion, transmission, background subtraction, scaling, and reduction.


In [ ]:
background_path = preprocess_usaxs_acquisition(
    DATA_DIR, BACKGROUND_SCAN_SET, PREPROCESSED_DIR, overwrite=OVERWRITE_PREPROCESSED
)
sample_paths = [
    preprocess_usaxs_acquisition(
        DATA_DIR, scan_set, PREPROCESSED_DIR, overwrite=OVERWRITE_PREPROCESSED
    )
    for scan_set in SAMPLE_SCAN_SETS[:MAX_MEASUREMENTS]
]

print(f"Python: {sys.executable}")
print(f"MoDaCor: {modacor.__version__}")
print(f"Background: {background_path.name}")
print(f"Samples: {len(sample_paths)}")


## Inspect the correction graph


In [ ]:
pipeline = Pipeline.from_yaml_file(yaml_file=PIPELINE_PATH)
pipeline.prepare()
display(Markdown(f"```mermaid\n{pipeline.to_mermaid(direction='TD')}\n```"))
print(f"USAXS pipeline: {len(pipeline.graph)} steps")


## Start or reuse a local runtime


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
server = LocalRuntimeServer(
    host=SERVER_HOST,
    port=SERVER_PORT,
    log_path=OUTPUT_DIR / "modacor_server.log",
    environment={"HDF5_PLUGIN_PATH": hdf5plugin.PLUGINS_PATH},
)
client = server.start()
atexit.register(server.stop)
print(f"{'Started' if server.launched else 'Reusing'} runtime at {client.base_url}")


## Process the three sample acquisitions

The background source stays registered while only the sample source changes, allowing the runtime to use its dependency graph for partial reruns.


In [ ]:
session = client.replace_session(
    "i22-usaxs-server-batch",
    name="I22 USAXS server batch",
    pipeline_yaml_path=str(PIPELINE_PATH),
    trace=TRACE,
)
session.register_sources(
    {"ref": "sample", "type": "hdf", "location": str(sample_paths[0])},
    {"ref": "background", "type": "hdf", "location": str(background_path)},
)

results = []
for sample_path in sample_paths:
    output = OUTPUT_DIR / f"{sample_path.stem}_result.h5"
    session.register_source({"ref": "sample", "type": "hdf", "location": str(sample_path)})
    result = session.process(
        mode="auto",
        changed_sources=["sample"],
        run_name=sample_path.stem,
        rollback_snapshot=False,
        write_hdf={"path": str(output), "data_paths": OUTPUT_DATA_PATHS},
    )
    results.append({"sample": sample_path, "output": output, "result": result})
    print(f"{sample_path.name}: {result['status']} ({result['effective_mode']})")

display(JSON(session.latest_error()))


## Compare reduced curves and diagnostics

The stored Q values are the actual weighted mean coordinates of contributing points, not geometric bin centres. Negative intensities remain visible because they are meaningful background-subtraction diagnostics.


In [ ]:
fig, (ax_wings, ax_curve, ax_count) = plt.subplots(3, 1, figsize=(8, 11))
for item in results:
    with h5py.File(item["output"], "r") as source:
        result_root = source["/processing/result"]
        run = result_root[next(iter(result_root))]
        q = run["pooled/Q/signal"][()]
        intensity = run["pooled/signal/signal"][()]
        count = run["pooled/bin_count/signal"][()]
        signed_q = run["SLF/signed_Q/signal"][()]
        both_wings = run["SLF/both_wings_signal/signal"][()]
        transmission = float(np.asarray(run["sample_integral/signal/signal"][()]).squeeze())
        rear_transmission = float(np.asarray(run["sample_rear_integral/signal/signal"][()]).squeeze())
        low_center = float(np.asarray(run["SLF/beam_center/signal"][()]).squeeze())
        high_center = float(np.asarray(run["SHF/beam_center/signal"][()]).squeeze())
        scales = [
            float(np.asarray(run[f"{key}/scale_factor/signal"][()]).squeeze())
            for key in ("SLR", "SHF", "SHR")
        ]
    label = item["sample"].stem
    order = np.argsort(signed_q, kind="stable")
    ax_wings.plot(signed_q[order], both_wings[order], label=label)
    ax_curve.plot(q, intensity, label=label)
    ax_count.plot(q, count, label=label)
    print(
        f"{label}: T front/rear={transmission:.6f}/{rear_transmission:.6f}, "
        f"centres={low_center:.4f}/{high_center:.4f} urad, "
        f"scales rear-low/front-high/rear-high={scales[0]:.4g}/{scales[1]:.4g}/{scales[2]:.4g}"
    )

ax_wings.axvline(0, color="0.6", linewidth=0.8)
ax_wings.axhline(0, color="0.6", linewidth=0.8)
ax_wings.set_xscale("symlog", linthresh=8e-4)
ax_wings.set_yscale("symlog", linthresh=1e-5)
ax_wings.set_xlabel(r"signed $Q$ / nm$^{-1}$")
ax_wings.set_ylabel("SLF both-wing intensity")
ax_wings.legend()
ax_curve.axhline(0, color="0.6", linewidth=0.8)
ax_curve.set_xscale("log")
ax_curve.set_xlabel(r"$Q$ / nm$^{-1}$")
ax_curve.set_ylabel("corrected intensity")
ax_curve.legend()
ax_count.set_xscale("log")
ax_count.set_yscale("log")
ax_count.set_xlabel(r"$Q$ / nm$^{-1}$")
ax_count.set_ylabel("contributing points")
fig.tight_layout()


## Cleanup

This stops only a process launched by this notebook. A runtime already listening on the configured port is left alone.


In [ ]:
server.stop()
print("Stopped the notebook-owned runtime." if not client.is_ready() else "Left the external runtime running.")
